# 03 · Train on the ESCAPE benchmark

Same pipeline as notebook 01, on the **ESCAPE** train split. Self-contained.

> **Data:** point `TRAIN_URL` at your ESCAPE train CSV (columns `aa_seq, aa_len, AMP`).
> Options: a raw GitHub URL after you commit it under `data/raw/escape/`, a public
> link, or upload to Colab and use a local path like `'escape_train.csv'`.

In [ ]:
# === Setup: install a transformers version with the classic BERT tokenizer ===
# (Colab's bundled transformers is too new and mishandles ProtBERT-BFD.)
%pip install -q transformers==4.46.3 "tokenizers>=0.20,<0.21" sentencepiece

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)   # expect 'cuda' — set Runtime > Change runtime type > GPU

## Config

In [ ]:
MODEL_NAME   = 'Rostlab/prot_bert_bfd'
MAX_LEN      = 200
NUM_EPOCHS   = 15
LR           = 5e-5
BATCH_SIZE   = 1
GRAD_ACCUM   = 64
WEIGHT_DECAY = 0.1
FP16         = True
SEED         = 0

# TODO: set to your ESCAPE train split
TRAIN_URL  = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/escape/escape_train.csv'
MODEL_DIR  = 'models/amp_bert_escape'
OUTPUT_DIR = 'results/escape_train'

In [ ]:
# === Core logic (self-contained; mirrors src/amp_bert/) ===
import pandas as pd, numpy as np
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             matthews_corrcoef, roc_auc_score)


def load_dataset(path, shuffle=False, random_state=0):
    df = pd.read_csv(path, index_col=0)
    return df.sample(frac=1, random_state=random_state) if shuffle else df


class AmpDataset(Dataset):
    """Tokenised view over an AMP DataFrame (columns: aa_seq, AMP)."""
    def __init__(self, df, tokenizer_name=MODEL_NAME, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_name, do_lower_case=False, use_fast=False)
        self.max_len = max_len
        self.seqs = list(self.df['aa_seq'])
        self.labels = list(self.df['AMP'].astype(int))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = ' '.join(''.join(self.seqs[idx].split()))
        ids = self.tokenizer(seq, truncation=True, padding='max_length',
                             max_length=self.max_len)
        sample = {k: torch.tensor(v) for k, v in ids.items()}
        sample['labels'] = torch.tensor(self.labels[idx])
        return sample


def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = logits.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    m = {'accuracy': accuracy_score(labels, preds), 'f1': f1,
         'precision': precision, 'recall': recall,
         'mcc': matthews_corrcoef(labels, preds)}
    if len(np.unique(labels)) == 2:
        z = logits - logits.max(axis=-1, keepdims=True)
        probs = np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)
        m['roc_auc'] = roc_auc_score(labels, probs[:, 1])
    return m


def model_init():   # must be zero-arg: Trainer passes `trial` to 1-arg fns
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


def make_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir, num_train_epochs=NUM_EPOCHS,
        learning_rate=LR, per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM, weight_decay=WEIGHT_DECAY,
        warmup_steps=0, logging_steps=100, eval_strategy='no',
        save_strategy='no', fp16=FP16, seed=SEED, report_to='none')

## Load data & train

In [ ]:
df = load_dataset(TRAIN_URL, shuffle=True, random_state=SEED)
print(df.shape); df.head()

In [ ]:
train_dataset = AmpDataset(df)
trainer = Trainer(model_init=model_init, args=make_training_args(OUTPUT_DIR),
                  train_dataset=train_dataset, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
trainer.save_model(MODEL_DIR)
print('saved to', MODEL_DIR)